# Dataset Metadata Inspector
Checks what geographic/temporal metadata is physically available in PANGAEA dataset files on disk.
Relevant for Copernicus-FM integration, which needs: **lon, lat, time_delta (days since 1970-01-01), patch_area**.

Datasets covered: HLS Burn Scars, MADOS, PASTIS-R, Sen1Floods11, Five Billion Pixels, DynamicEarthNet, CropTypeMapping-SS, SpaceNet7, AI4SmallFarms

In [22]:
import json
import math
import os
from datetime import datetime
from pathlib import Path

import numpy as np
import rasterio
from pyproj import Transformer

DATA_ROOT = Path("/dss/dsstbyfs02/pn49cu/pn49cu-dss-0001/pangaea-datasets")
EPOCH = datetime(1970, 1, 1)


def tif_geo(path):
    """Return (lon, lat, patch_area_km2, crs_str) from a GeoTIFF."""
    with rasterio.open(path) as src:
        crs = src.crs
        if crs is None:
            return None, None, None, "no CRS"
        cx = (src.bounds.left + src.bounds.right) / 2
        cy = (src.bounds.bottom + src.bounds.top) / 2
        if crs.is_geographic:
            lon, lat = cx, cy
        else:
            t = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
            lon, lat = t.transform(cx, cy)
        if not crs.is_geographic:
            area = abs(src.res[0] * src.res[1]) * src.width * src.height / 1e6
        else:
            # approximate using haversine on bounds
            from pyproj import Geod
            geod = Geod(ellps="WGS84")
            _, _, w = geod.inv(src.bounds.left, cy, src.bounds.right, cy)
            _, _, h = geod.inv(cx, src.bounds.bottom, cx, src.bounds.top)
            area = (w * h) / 1e6
        return lon, lat, area, str(crs)


def days_since_epoch(date_str, fmt="%Y-%m-%d"):
    return (datetime.strptime(date_str, fmt) - EPOCH).days


def tile_xyz_to_lonlat(x, y, z):
    """Convert XYZ slippy map tile to (lon, lat) of tile center."""
    n = 2 ** z
    lon = (x + 0.5) / n * 360 - 180
    lat = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (y + 0.5) / n))))
    return lon, lat


def print_result(name, lon, lat, area, time_delta, notes=""):
    lon_s  = f"{lon:.3f}"       if lon       is not None else "N/A"
    lat_s  = f"{lat:.3f}"       if lat       is not None else "N/A"
    area_s = f"{area:.2f} km²"  if area      is not None else "N/A"
    time_s = str(time_delta)     if time_delta is not None else "N/A"
    print(f"[{name}]")
    print(f"  lon={lon_s}  lat={lat_s}  area={area_s}  time_delta={time_s}")
    if notes:
        print(f"  notes: {notes}")
    print()

## 1. HLS Burn Scars
TIF files. CRS = UTM (projected). Date encoded in filename as `YYYYDDD`.

In [23]:
root = DATA_ROOT / "hlsburnscars"
sample = next((root / "training").glob("*_merged.tif"))
print("Sample:", sample.name)

lon, lat, area, crs = tif_geo(sample)

# filename: subsetted_512x512_HLS.S30.T13SFA.2020154.v1.4_merged.tif
date_str = sample.stem.split(".")[3]   # e.g. '2020154'
date = datetime.strptime(date_str, "%Y%j")
time_delta = (date - EPOCH).days

print_result("HLS Burn Scars", lon, lat, area, time_delta,
             notes=f"CRS={crs}, date={date.date()} (from filename YYYYDDD)")

Sample: subsetted_512x512_HLS.S30.T13SFA.2020154.v1.4_merged.tif
[HLS Burn Scars]
  lon=-103.202  lat=36.951  area=235.93 km²  time_delta=18415
  notes: CRS=PROJCS["UTM Zone 13, Northern Hemisphere",GEOGCS["Unknown datum based upon the WGS 84 ellipsoid",DATUM["Not specified (based on WGS 84 spheroid)",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-105],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]], date=2020-06-02 (from filename YYYYDDD)



## 2. MADOS
Per-band TIF files. Files have **no geotransform** (NotGeoreferencedWarning). No timestamp metadata.

In [24]:
import warnings
root = DATA_ROOT / "mados"
sample = next(root.rglob("*.tif"))
print("Sample:", sample)

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    with rasterio.open(sample) as src:
        crs = src.crs
        has_transform = src.transform != rasterio.transform.IDENTITY
        print(f"  CRS: {crs}")
        print(f"  Has real geotransform: {has_transform}")
        print(f"  Transform: {src.transform}")
    if w:
        print(f"  Warning: {w[0].message}")

# Scene name has no date; check if scene folder name is informative
scene_name = sample.parts[-3]  # e.g. 'Scene_84'
print(f"  Scene folder: {scene_name} — no date info in name")
print()
print_result("MADOS", None, None, None, None,
             notes="TIFs have no geotransform → lon/lat/area not available from files. No timestamp source found.")

Sample: /dss/dsstbyfs02/pn49cu/pn49cu-dss-0001/pangaea-datasets/mados/Scene_84/60/Scene_84_L2R_rhorc_443_29.tif
  CRS: EPSG:4326
  Has real geotransform: False
  Transform: | 1.00, 0.00, 0.00|
| 0.00, 1.00, 0.00|
| 0.00, 0.00, 1.00|
  Scene folder: Scene_84 — no date info in name

[MADOS]
  lon=N/A  lat=N/A  area=N/A  time_delta=N/A
  notes: TIFs have no geotransform → lon/lat/area not available from files. No timestamp source found.



## 3. PASTIS-R
NPY files. Rich `metadata.geojson` with patch geometry (→ lon/lat) and per-modality date lists.

In [25]:
root = DATA_ROOT / "pastis"

with open(root / "metadata.geojson") as f:
    gj = json.load(f)

# GeoJSON CRS is EPSG:2154 (Lambert 93) — must reproject centroid to WGS84
crs_name = gj.get("crs", {}).get("properties", {}).get("name", "unknown")
print(f"  GeoJSON CRS: {crs_name}")

feat = gj["features"][0]
props = feat["properties"]
geom  = feat["geometry"]

# Centroid in native CRS (EPSG:2154)
all_coords = []
for polygon in geom["coordinates"]:
    for ring in polygon:
        all_coords.extend(ring)
cx = sum(c[0] for c in all_coords) / len(all_coords)
cy = sum(c[1] for c in all_coords) / len(all_coords)
print(f"  Centroid in EPSG:2154: ({cx:.1f}, {cy:.1f})")

# Reproject to WGS84
t = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)
lon, lat = t.transform(cx, cy)

# dates from properties — use first S2 date as representative
s2_dates = props["dates-S2"]   # dict {"0": 20180924, ...}
first_date_int = list(s2_dates.values())[0]
date = datetime.strptime(str(first_date_int), "%Y%m%d")
time_delta = (date - EPOCH).days

# Patch area: PASTIS patches are 128x128 at 10m → ~1.64 km²
sample_npy = next((root / "DATA_S2").glob("*.npy")) if (root / "DATA_S2").exists() else None
if sample_npy:
    arr = np.load(sample_npy)  # shape (T, C, H, W)
    h, w = arr.shape[-2], arr.shape[-1]
    area = (10 * h * 10 * w) / 1e6   # 10m resolution
    print(f"  NPY shape: {arr.shape} → {h}x{w} px @ 10m")
else:
    area = None

print(f"  GeoJSON properties keys: {list(props.keys())}")
print(f"  S2 date range: {list(s2_dates.values())[0]} – {list(s2_dates.values())[-1]}")
print()
print_result("PASTIS-R", lon, lat, area, time_delta,
             notes=f"lon/lat reprojected EPSG:2154 → WGS84. {len(s2_dates)} S2 dates per patch.")

  GeoJSON CRS: urn:ogc:def:crs:EPSG::2154
  Centroid in EPSG:2154: (391976.1, 6956255.6)
  NPY shape: (43, 10, 128, 128) → 128x128 px @ 10m
  GeoJSON properties keys: ['Fold', 'ID_PATCH', 'N_Parcel', 'Parcel_Cover', 'TILE', 'dates-S1A', 'dates-S1D', 'dates-S2', 'id']
  S2 date range: 20180924 – 20191019

[PASTIS-R]
  lon=-1.263  lat=49.631  area=1.64 km²  time_delta=17798
  notes: lon/lat reprojected EPSG:2154 → WGS84. 43 S2 dates per patch.



## 4. Sen1Floods11
TIF files with EPSG:4326. `Sen1Floods11_Metadata.geojson` has `s2_date` per location.

In [26]:
root = DATA_ROOT / "sen1floods11" / "v1.1"

# Load GeoJSON — keyed by country name (Bolivia, Nigeria, Spain, USA, ...)
# Filenames use region/country prefix: e.g. Nigeria_952958, Spain_7604243, Mekong_977338
# "Mekong" is a region spanning multiple countries and has no direct GeoJSON entry.
# We pick a file whose prefix matches a GeoJSON location.
with open(root / "Sen1Floods11_Metadata.geojson") as f:
    gj = json.load(f)
geojson_locations = {f["properties"]["location"]: f["properties"] for f in gj["features"]}

s2_dir = root / "data" / "flood_events" / "HandLabeled" / "S2Hand"
sample = None
matched_props = None
for tif in sorted(s2_dir.glob("*.tif")):
    prefix = tif.stem.split("_")[0]   # e.g. 'Nigeria', 'Spain', 'Mekong'
    if prefix in geojson_locations:
        sample = tif
        matched_props = geojson_locations[prefix]
        break

if sample is None:
    print("No filename prefix matched a GeoJSON location entry — using first TIF without date")
    sample = next(s2_dir.glob("*.tif"))

print(f"  Sample: {sample.name}  prefix='{sample.stem.split('_')[0]}'")

lon, lat, area, crs = tif_geo(sample)

if matched_props:
    date_str = matched_props["s2_date"]   # '2018/02/15'
    date = datetime.strptime(date_str, "%Y/%m/%d")
    time_delta = (date - EPOCH).days
    date_note = f"s2_date={date_str} from GeoJSON"
else:
    time_delta = None
    date_note = "no GeoJSON match (region name not a country entry)"

print(f"  {date_note}")
print()
print_result("Sen1Floods11", lon, lat, area, time_delta,
             notes=f"CRS={crs}. Area via geodesic approx. Note: 'Mekong' has no GeoJSON entry (region ≠ country).")

  Sample: Bolivia_103757_S2Hand.tif  prefix='Bolivia'
  s2_date=2018/02/15 from GeoJSON

[Sen1Floods11]
  lon=-65.242  lat=-14.235  area=25.26 km²  time_delta=17577
  notes: CRS=EPSG:4326. Area via geodesic approx. Note: 'Mekong' has no GeoJSON entry (region ≠ country).



## 5. Five Billion Pixels
TIF files (Gaofen-2 imagery, China). Parquet catalog available.

In [27]:
root = DATA_ROOT / "fivebillionpixels"

# Label TIFs have no CRS — use image TIFs instead
sample = next((root / "train" / "images").glob("*.tif"))
print("Sample:", sample.name)

lon, lat, area, crs = tif_geo(sample)

# No timestamp source found in files or parquet (parquet needs pandas)
print()
print_result("Five Billion Pixels", lon, lat, area, None,
             notes=f"CRS={crs}. Images have geo; label TIFs do not. No timestamp source found.")

Sample: GF2_PMS2__L1A0000564692-MSS2_4_8.tif

[Five Billion Pixels]
  lon=N/A  lat=N/A  area=N/A  time_delta=N/A
  notes: CRS=no CRS. Images have geo; label TIFs do not. No timestamp source found.



/dss/dsshome1/08/di54zop/micromamba/envs/pangaea-bench/lib/python3.10/site-packages/rasterio/__init__.py:304: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


## 6. DynamicEarthNet
NPY files (Planet Fusion SR, 4-band, 1024x1024). No TIFs. Tile ID in folder name encodes XYZ web-mercator tile → lon/lat. Date encoded in filename as `YYYY-MM-DD.npy`.

In [28]:
root = DATA_ROOT / "dynamicen"

# Find a sample npy file: planet/<lat_zone>/<lon_zone>/<tile_xyz>/PF-SR/<date>.npy
sample = next(root.glob("planet/*/*/*/PF-SR/*.npy"))
print("Sample:", sample)

arr = np.load(sample)
h, w = arr.shape[-2], arr.shape[-1]
print(f"  Shape: {arr.shape}  ({h}x{w} px)")

# Tile ID is the folder 2 levels up from PF-SR, e.g. '6475_3361_13'
tile_folder = sample.parts[-3]  # e.g. '6475_3361_13'
tx, ty, tz = [int(v) for v in tile_folder.split("_")]
lon, lat = tile_xyz_to_lonlat(tx, ty, tz)

# Area: Planet Fusion SR is ~4.77m/px at zoom 13
# Actual resolution from tile: tile width in meters = 2*pi*R / 2^z / (tile_px)
R = 6378137
tile_size_m = 2 * math.pi * R / (2 ** tz)
res_m = tile_size_m / w  # meters per pixel
area = (res_m * w * res_m * h) / 1e6

# Date from filename
date_str = sample.stem  # '2018-01-01'
time_delta = days_since_epoch(date_str)

print(f"  Tile {tx},{ty},{tz} → lon={lon:.3f}, lat={lat:.3f}")
print(f"  Resolution: ~{res_m:.1f} m/px  →  area={area:.2f} km²")
print()
print_result("DynamicEarthNet", lon, lat, area, time_delta,
             notes=f"lon/lat from XYZ tile name. Date from filename. Res ~{res_m:.1f} m/px.")

Sample: /dss/dsstbyfs02/pn49cu/pn49cu-dss-0001/pangaea-datasets/dynamicen/planet/39N/10E-136N/5195_3388_13/PF-SR/2019-07-02.npy
  Shape: (4, 1024, 1024)  (1024x1024 px)
  Tile 5195,3388,13 → lon=48.318, lat=29.669
  Resolution: ~4.8 m/px  →  area=23.93 km²

[DynamicEarthNet]
  lon=48.318  lat=29.669  area=23.93 km²  time_delta=18079
  notes: lon/lat from XYZ tile name. Date from filename. Res ~4.8 m/px.



## 7. CropTypeMapping-SS
NPZ files with arrays (planet/s1/s2/s2_cloudmask). Per-patch JSON sidecar has `dates` list. No direct geo coordinates in files.

In [29]:
root = DATA_ROOT / "croptypemapping" / "southsudan"

# Load a sample NPZ
sample_npz = next((root / "npy").glob("*.npz"))
npz = np.load(sample_npz, allow_pickle=True)
print(f"NPZ keys: {npz.files}")
for k in npz.files:
    print(f"  {k}: shape={npz[k].shape}, dtype={npz[k].dtype}")

# Corresponding JSON for dates
patch_id = sample_npz.stem  # e.g. 'southsudan_002446'
json_path = root / "planet" / f"planet_{patch_id}.json"
if json_path.exists():
    with open(json_path) as f:
        meta = json.load(f)
    dates = meta["dates"]
    first_date = dates[0]
    time_delta = days_since_epoch(first_date)
    date_note = f"{len(dates)} dates, first={first_date}"
else:
    time_delta = None
    date_note = "JSON not found"

# Check CSV for any geo info
csv_path = root / "list_eval_partition.csv"
geo_note = "no geo coordinates in files"
if csv_path.exists():
    lines = csv_path.read_text().splitlines()
    print(f"\nCSV header: {lines[0]}")
    print(f"CSV sample: {lines[1]}")
    header = lines[0].lower()
    if "lon" in header or "lat" in header or "x" in header:
        geo_note = "CSV may contain coordinates — check header above"

print()
print_result("CropTypeMapping-SS", None, None, None, time_delta,
             notes=f"{date_note}. {geo_note}.")

NPZ keys: ['planet', 's1', 's2', 's2_cloudmask']


  planet: shape=(4, 212, 212, 67), dtype=uint16
  s1: shape=(3, 64, 64, 47), dtype=float64
  s2: shape=(10, 64, 64, 52), dtype=float64
  s2_cloudmask: shape=(64, 64, 52), dtype=float64

CSV header: ,id,partition
CSV sample: 0,006761,0

[CropTypeMapping-SS]
  lon=N/A  lat=N/A  area=N/A  time_delta=17287
  notes: 67 dates, first=2017-05-01. no geo coordinates in files.



## 8. SpaceNet7
TIF files. CRS = EPSG:3857 (Web Mercator). `metadata_train.json` has per-AOI timestamp lists.

In [ ]:
rows = [
    # (dataset,            lon_src,                  area_src,               timestamp_src)
    ("HLS Burn Scars",   "✅ TIF bounds (UTM)",      "✅ TIF res×size",       "✅ filename YYYYDDD"),
    ("MADOS",            "❌ not georeferenced",     "❌ not georeferenced",  "❌ none"),
    ("PASTIS-R",         "✅ GeoJSON (EPSG:2154→WGS84)", "✅ NPY shape × 10m", "✅ GeoJSON dates-S2"),
    ("Sen1Floods11",     "✅ TIF bounds (WGS84)",    "✅ geodesic approx",    "✅ GeoJSON s2_date"),
    ("Five Billion Px",  "❌ not georeferenced",     "❌ not georeferenced",  "❌ none"),
    ("DynamicEarthNet",  "✅ XYZ tile name",         "✅ tile size / zoom",   "✅ filename YYYY-MM-DD"),
    ("CropTypeMapping",  "❌ none in files",         "❌ none in files",      "✅ JSON sidecar dates"),
    ("SpaceNet7",        "✅ TIF bounds (EPSG:3857)","✅ TIF res×size",       "✅ metadata_train.json"),
    ("AI4SmallFarms",    "✅ TIF bounds (UTM)",      "✅ TIF res×size",       "❌ none"),
]

print(f"{'Dataset':<20} {'lon/lat':^8} {'area':^8} {'timestamp':^10}   details")
print("-" * 110)
for ds, lon_src, area_src, ts_src in rows:
    has_lonlat = "✅" if lon_src.startswith("✅") else "❌"
    has_area   = "✅" if area_src.startswith("✅") else "❌"
    has_ts     = "✅" if ts_src.startswith("✅") else "❌"
    print(f"{ds:<20} {has_lonlat:^8} {has_area:^8} {has_ts:^10}   lon/lat: {lon_src} | ts: {ts_src}")

## 9. AI4SmallFarms
TIF files. CRS = UTM (projected). No timestamp metadata found.

In [31]:
root = DATA_ROOT / "ai4smallfarms"
sample = next(root.rglob("*.tif"))
print("Sample:", sample.name)

lon, lat, area, crs = tif_geo(sample)

meta_files = list(root.rglob("*.json")) + list(root.rglob("*.csv")) + list(root.rglob("*.geojson"))
print(f"  Sidecar files: {[str(p.relative_to(root)) for p in meta_files[:5]] or 'none'}")
print()
print_result("AI4SmallFarms", lon, lat, area, None,
             notes=f"CRS={crs}. No timestamp source in files.")

Sample: 24_vietnam.tif
  Sidecar files: none

[AI4SmallFarms]
  lon=106.166  lat=21.007  area=23.63 km²  time_delta=N/A
  notes: CRS=EPSG:32648. No timestamp source in files.



## Summary

In [32]:
rows = [
    # (dataset,          lon_src,            lat_src,            area_src,             timestamp_src)
    ("HLS Burn Scars",   "TIF bounds (UTM)",  "TIF bounds (UTM)",  "TIF res×size",        "filename YYYYDDD"),
    ("MADOS",            "❌ not georef'd",   "❌ not georef'd",   "❌ not georef'd",     "❌ none"),
    ("PASTIS-R",         "metadata.geojson",  "metadata.geojson",  "NPY shape × 10m res", "metadata.geojson dates-S2"),
    ("Sen1Floods11",     "TIF bounds (WGS84)","TIF bounds (WGS84)","TIF res×size (geodesic)","metadata.geojson s2_date"),
    ("Five Billion Px",  "TIF bounds",        "TIF bounds",        "TIF res×size",        "❌ none found"),
    ("DynamicEarthNet",  "XYZ tile name",     "XYZ tile name",     "tile size / zoom",    "filename YYYY-MM-DD"),
    ("CropTypeMapping",  "❌ none",           "❌ none",           "❌ none",             "JSON sidecar dates list"),
    ("SpaceNet7",        "TIF bounds (3857)", "TIF bounds (3857)", "TIF res×size",        "metadata_train.json"),
    ("AI4SmallFarms",    "TIF bounds (UTM)",  "TIF bounds (UTM)",  "TIF res×size",        "❌ none"),
]

print(f"{'Dataset':<20} {'lon/lat':^8} {'area':^8} {'timestamp':^8}   sources")
print("-" * 100)
for ds, lon_src, lat_src, area_src, ts_src in rows:
    has_lonlat = "✅" if not lon_src.startswith("❌") else "❌"
    has_area   = "✅" if not area_src.startswith("❌") else "❌"
    has_ts     = "✅" if not ts_src.startswith("❌") else "❌"
    print(f"{ds:<20} {has_lonlat:^8} {has_area:^8} {has_ts:^8}   lon/lat: {lon_src} | ts: {ts_src}")

Dataset              lon/lat    area   timestamp   sources
----------------------------------------------------------------------------------------------------
HLS Burn Scars          ✅        ✅        ✅       lon/lat: TIF bounds (UTM) | ts: filename YYYYDDD
MADOS                   ❌        ❌        ❌       lon/lat: ❌ not georef'd | ts: ❌ none
PASTIS-R                ✅        ✅        ✅       lon/lat: metadata.geojson | ts: metadata.geojson dates-S2
Sen1Floods11            ✅        ✅        ✅       lon/lat: TIF bounds (WGS84) | ts: metadata.geojson s2_date
Five Billion Px         ✅        ✅        ❌       lon/lat: TIF bounds | ts: ❌ none found
DynamicEarthNet         ✅        ✅        ✅       lon/lat: XYZ tile name | ts: filename YYYY-MM-DD
CropTypeMapping         ❌        ❌        ✅       lon/lat: ❌ none | ts: JSON sidecar dates list
SpaceNet7               ✅        ✅        ✅       lon/lat: TIF bounds (3857) | ts: metadata_train.json
AI4SmallFarms           ✅        ✅        ❌       